# Tamil Nadu Electricity Demand-Supply Forecasting — RAG Pipeline
## Complete 6-Stage Modular Implementation

This notebook implements the complete Retrieval-Augmented Generation (RAG) system for electricity policy and resource adequacy planning in Tamil Nadu.

### End-to-End RAG Architecture:
1. **Stage 1**: Document Ingestion & Page-Level PDF Text Extraction
2. **Stage 2**: Page-Level Deterministic Text Cleaning & Quality Diagnostics
3. **Stage 3**: Token-Aware Semantic Text Chunking & Lineage Tracking
4. **Stage 4**: Vector Embedding Generation & FAISS Vector Database Creation
5. **Stage 5**: Dynamic Forecast-Based Query Construction & Similarity Retrieval
6. **Stage 6**: Gemini LLM Integration & Grounded Energy Recommendation Generation

---

---
## Stage 1: Document Ingestion & Page-Level PDF Text Extraction

In [1]:
import os
import json
import re
import time
import logging
from pathlib import Path
from typing import List, Dict, Any, Tuple
from dotenv import load_dotenv
from pypdf import PdfReader
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("RAG_Pipeline")

# Define repository & data paths
BASE_DIR = Path.cwd()
KB_DIR = BASE_DIR / "knowledge_base" if (BASE_DIR / "knowledge_base").exists() else BASE_DIR / "RAG" / "knowledge_base"
PROCESSED_DIR = BASE_DIR / "processed" if (BASE_DIR / "processed").exists() else BASE_DIR / "RAG" / "processed"
VECTOR_DB_DIR = BASE_DIR / "vector_db" if (BASE_DIR / "vector_db").exists() else BASE_DIR / "RAG" / "vector_db"

PAGES_OUTPUT_FILE = PROCESSED_DIR / "extracted_pages.json"
CLEANED_PAGES_FILE = PROCESSED_DIR / "cleaned_pages.json"
CHUNKS_OUTPUT_FILE = PROCESSED_DIR / "chunks.json"
PREVIEW_TXT_FILE = PROCESSED_DIR / "chunks_preview.txt"

FAISS_INDEX_FILE = VECTOR_DB_DIR / "energy_knowledge_base.faiss"
METADATA_MAPPING_FILE = VECTOR_DB_DIR / "chunk_metadata.json"
CONFIG_FILE = VECTOR_DB_DIR / "embedding_config.json"

# Load environment variables from .env
load_dotenv(dotenv_path=BASE_DIR / ".env")

print(f"Knowledge Base Path : {KB_DIR.resolve()}")
print(f"Processed Dir Path  : {PROCESSED_DIR.resolve()}")
print(f"Vector DB Path      : {VECTOR_DB_DIR.resolve()}")

Knowledge Base Path : D:\Final Year Project 2027\RAG\knowledge_base
Processed Dir Path  : D:\Final Year Project 2027\RAG\processed
Vector DB Path      : D:\Final Year Project 2027\RAG\vector_db


### 1.1 Knowledge Base Discovery

In [2]:
def discover_pdf_documents(base_dir: Path) -> List[Path]:
    if not base_dir.exists():
        raise FileNotFoundError(f"Knowledge base directory does not exist: {base_dir}")
    return sorted(
        list(base_dir.rglob("*.pdf")),
        key=lambda p: (p.parent.name.lower(), p.name.lower())
    )

discovered_pdfs = discover_pdf_documents(KB_DIR)
print(f"Total PDF Documents Discovered: {len(discovered_pdfs)}\n")
for idx, pdf in enumerate(discovered_pdfs, start=1):
    category = pdf.parent.name
    size_mb = pdf.stat().st_size / (1024 * 1024)
    print(f"  {idx}. [{category:<5}] {pdf.name:<45} ({size_mb:6.2f} MB)")

Total PDF Documents Discovered: 9

  1. [India] annual_reports.pdf                            (  6.33 MB)
  2. [India] grid_code.pdf                                 (  2.44 MB)
  3. [India] national_electricity_policy-1.pdf             (  1.65 MB)
  4. [India] nep-vol-1.pdf                                 ( 18.48 MB)
  5. [India] nep-vol-2.pdf                                 ( 27.34 MB)
  6. [India] Renewable_integration.pdf                     (  3.34 MB)
  7. [Tn   ] Tamil_Nadu_Resource_Adequacy_Report_2026.pdf  (  0.95 MB)
  8. [Tn   ] Tamilnadu_adequate_plan.pdf                   (  2.66 MB)
  9. [Tn   ] tamilnadu_energy_department.pdf               (  1.97 MB)


### 1.2 Page-by-Page PDF Extraction

In [3]:
def extract_pages_from_pdf(pdf_path: Path, min_char_threshold: int = 50) -> Dict[str, Any]:
    category = pdf_path.parent.name
    filename = pdf_path.name
    doc_records, empty_pages, short_pages, extraction_errors = [], [], [], []
    
    try:
        reader = PdfReader(str(pdf_path))
        total_pages = len(reader.pages)
    except Exception as e:
        logger.error(f"Failed to open/parse PDF {filename}: {e}")
        return {
            "source": filename, "category": category, "total_pages": 0, "extracted_count": 0,
            "empty_pages": [], "short_pages": [], "extraction_errors": [{"page": 0, "error": str(e)}],
            "records": [], "total_chars": 0
        }
    
    total_chars = 0
    extracted_count = 0
    
    for page_num, page in enumerate(reader.pages, start=1):
        try:
            raw_text = page.extract_text() or ""
            cleaned_text = raw_text.strip()
            char_len = len(cleaned_text)
            
            if char_len == 0:
                empty_pages.append(page_num)
            elif char_len < min_char_threshold:
                short_pages.append({"page": page_num, "char_count": char_len, "preview": cleaned_text[:40]})
                extracted_count += 1
            else:
                extracted_count += 1
                
            total_chars += char_len
            doc_records.append({"source": filename, "category": category, "page": page_num, "text": cleaned_text})
        except Exception as pe:
            logger.warning(f"Error extracting text from {filename} page {page_num}: {pe}")
            extraction_errors.append({"page": page_num, "error": str(pe)})
            doc_records.append({"source": filename, "category": category, "page": page_num, "text": ""})
            
    return {
        "source": filename, "category": category, "total_pages": total_pages,
        "extracted_count": extracted_count, "empty_pages": empty_pages,
        "short_pages": short_pages, "extraction_errors": extraction_errors,
        "records": doc_records, "total_chars": total_chars
    }

### 1.3 Stage 1 Execution & Output Generation

In [4]:
all_page_records = []
doc_summaries = []

for pdf_path in discovered_pdfs:
    res = extract_pages_from_pdf(pdf_path, min_char_threshold=50)
    all_page_records.extend(res["records"])
    doc_summaries.append({
        "source": res["source"], "category": res["category"], "total_pages": res["total_pages"],
        "extracted_pages": res["extracted_count"], "empty_pages_count": len(res["empty_pages"]),
        "short_pages_count": len(res["short_pages"]), "errors_count": len(res["extraction_errors"]),
        "total_chars": res["total_chars"],
        "avg_chars_per_page": round(res["total_chars"] / max(res["total_pages"], 1), 1)
    })

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
with open(PAGES_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_page_records, f, indent=2, ensure_ascii=False)
print(f"Saved {len(all_page_records)} page records to {PAGES_OUTPUT_FILE.name}")

Saved 1725 page records to extracted_pages.json


---
## Stage 2: Page-Level Text Cleaning & Quality Verification

In [5]:
def clean_page_text(raw_text: str) -> str:
    if not raw_text:
        return ""
    text = raw_text
    text = re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', ' ', text)
    text = re.sub(r'([a-zA-Z]{2,})-\s*\n\s*([a-zA-Z]{2,})', r'\1\2', text)
    text = text.replace('\ufffd', ' ')
    
    raw_lines = [line.strip() for line in text.split('\n')]
    cleaned_paragraphs = []
    current_para = []
    
    for line in raw_lines:
        if not line:
            if current_para:
                cleaned_paragraphs.append(" ".join(current_para))
                current_para = []
            continue
            
        is_heading_or_list = bool(
            re.match(r'^(?:\d+(?:\.\d+)*|\[[a-zA-Z0-9]+\]|[•\-\*•])\s+', line) or
            re.match(r'^(?:SECTION|CHAPTER|TABLE|ANNEXURE|DISCLAIMER|CONTENTS|EXECUTIVE SUMMARY|INTRODUCTION)\b', line, re.I)
        )
        
        if is_heading_or_list and current_para:
            cleaned_paragraphs.append(" ".join(current_para))
            current_para = [line]
        else:
            current_para.append(line)
            
    if current_para:
        cleaned_paragraphs.append(" ".join(current_para))
        
    cleaned_text = "\n\n".join(cleaned_paragraphs)
    lines = [re.sub(r'[ \t]+', ' ', l).strip() for l in cleaned_text.split('\n')]
    cleaned_text = "\n".join(lines)
    return re.sub(r'\n{3,}', '\n\n', cleaned_text).strip()

In [6]:
cleaned_records = []
for item in all_page_records:
    raw_txt = item.get("text", "")
    clean_txt = clean_page_text(raw_txt)
    record = {
        "source": item["source"],
        "category": item["category"],
        "page": item["page"],
        "text": raw_txt,
        "cleaned_text": clean_txt
    }
    cleaned_records.append(record)

with open(CLEANED_PAGES_FILE, "w", encoding="utf-8") as f:
    json.dump(cleaned_records, f, indent=2, ensure_ascii=False)
print(f"Saved {len(cleaned_records)} cleaned page records to {CLEANED_PAGES_FILE.name}")

Saved 1725 cleaned page records to cleaned_pages.json


---
## Stage 3: Token-Aware Semantic Text Chunking

In [7]:
try:
    import tiktoken
    tokenizer = tiktoken.get_encoding("cl100k_base")
    def count_tokens(text: str) -> int:
        return len(tokenizer.encode(text))
except ImportError:
    def count_tokens(text: str) -> int:
        return int(len(text.split()) * 1.3)


def split_text_into_semantic_units(text: str) -> List[str]:
    if not text:
        return []
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    units = []
    sentence_endings = re.compile(r'(?<=[.!?])\s+')
    for para in paragraphs:
        if count_tokens(para) <= 400:
            units.append(para)
        else:
            sentences = sentence_endings.split(para)
            for s in sentences:
                s_clean = s.strip()
                if s_clean:
                    units.append(s_clean)
    return units


def create_semantic_chunks_for_document(
    doc_pages: List[Dict[str, Any]],
    target_tokens: int = 900,
    max_tokens: int = 1100,
    overlap_tokens: int = 120
) -> List[Dict[str, Any]]:
    if not doc_pages:
        return []
    
    source = doc_pages[0]["source"]
    category = doc_pages[0]["category"]
    clean_stem = re.sub(r'[^a-zA-Z0-9]', '_', Path(source).stem)
    
    atomic_units: List[Tuple[str, int]] = []
    for p_rec in doc_pages:
        pg_num = p_rec["page"]
        pg_text = p_rec.get("cleaned_text", "").strip()
        if not pg_text:
            continue
        u_list = split_text_into_semantic_units(pg_text)
        for u in u_list:
            atomic_units.append((u, pg_num))
            
    if not atomic_units:
        return []
    
    chunks = []
    current_units: List[Tuple[str, int]] = []
    current_token_count = 0
    chunk_idx = 1
    
    for u_text, u_page in atomic_units:
        u_tokens = count_tokens(u_text)
        
        if u_tokens > max_tokens:
            sub_clauses = re.split(r'(?<=[;,])\s+', u_text)
            for sc in sub_clauses:
                sc_tokens = count_tokens(sc)
                if current_token_count + sc_tokens > target_tokens and current_units:
                    chunk_rec = build_chunk_record(current_units, source, category, clean_stem, chunk_idx, overlap_tokens)
                    chunks.append(chunk_rec["chunk"])
                    chunk_idx += 1
                    current_units = chunk_rec["overlap_units"]
                    current_token_count = sum(count_tokens(txt) for txt, _ in current_units)
                current_units.append((sc, u_page))
                current_token_count += sc_tokens
            continue
            
        if current_token_count + u_tokens > target_tokens and current_units:
            chunk_rec = build_chunk_record(current_units, source, category, clean_stem, chunk_idx, overlap_tokens)
            chunks.append(chunk_rec["chunk"])
            chunk_idx += 1
            current_units = chunk_rec["overlap_units"]
            current_token_count = sum(count_tokens(txt) for txt, _ in current_units)
            
        current_units.append((u_text, u_page))
        current_token_count += u_tokens
        
    if current_units:
        chunk_str = "\n\n".join(txt for txt, _ in current_units)
        t_cnt = count_tokens(chunk_str)
        pages_covered = sorted(list(set(pg for _, pg in current_units)))
        page_start = pages_covered[0]
        chunk_id = f"{category}_{clean_stem}_p{page_start}_c{chunk_idx}"
        chunks.append({
            "chunk_id": chunk_id,
            "source": source,
            "category": category,
            "page": page_start,
            "pages": pages_covered,
            "token_count": t_cnt,
            "char_count": len(chunk_str),
            "word_count": len(chunk_str.split()),
            "text": chunk_str
        })
        
    return chunks


def build_chunk_record(units: List[Tuple[str, int]], source: str, category: str, clean_stem: str, chunk_idx: int, overlap_tokens: int):
    chunk_str = "\n\n".join(txt for txt, _ in units)
    t_cnt = count_tokens(chunk_str)
    pages_covered = sorted(list(set(pg for _, pg in units)))
    page_start = pages_covered[0]
    chunk_id = f"{category}_{clean_stem}_p{page_start}_c{chunk_idx}"
    
    chunk_dict = {
        "chunk_id": chunk_id,
        "source": source,
        "category": category,
        "page": page_start,
        "pages": pages_covered,
        "token_count": t_cnt,
        "char_count": len(chunk_str),
        "word_count": len(chunk_str.split()),
        "text": chunk_str
    }
    
    overlap_units = []
    accum_tokens = 0
    for txt, pg in reversed(units):
        tk = count_tokens(txt)
        if accum_tokens + tk <= overlap_tokens:
            overlap_units.insert(0, (txt, pg))
            accum_tokens += tk
        else:
            break
            
    return {"chunk": chunk_dict, "overlap_units": overlap_units}

In [8]:
docs_map = {}
for p in cleaned_records:
    docs_map.setdefault(p["source"], []).append(p)

all_chunks = []
for src, p_list in docs_map.items():
    p_list_sorted = sorted(p_list, key=lambda x: x["page"])
    doc_chunks = create_semantic_chunks_for_document(p_list_sorted, target_tokens=900, overlap_tokens=120)
    all_chunks.extend(doc_chunks)

with open(CHUNKS_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)
print(f"Generated and saved {len(all_chunks)} chunks to {CHUNKS_OUTPUT_FILE.name}")

Generated and saved 1155 chunks to chunks.json


---
## Stage 4: Vector Embedding Generation & FAISS Vector Database Creation

In [9]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

model = SentenceTransformer(MODEL_NAME)
texts = [c["text"] for c in all_chunks]

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(embeddings)

VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(FAISS_INDEX_FILE))

with open(METADATA_MAPPING_FILE, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print(f"FAISS Index created with {index.ntotal} vectors saved to {FAISS_INDEX_FILE.name}")

2026-08-31 11:56:20,758 [INFO] No device provided, using cpu


2026-08-31 11:56:21,083 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-08-31 11:56:21,098 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


2026-08-31 11:56:21,334 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


2026-08-31 11:56:21,336 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


...

2026-08-31 11:56:26,138 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"


2026-08-31 11:56:26,149 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


2026-08-31 11:56:26,378 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/2_Normalize/config.json "HTTP/1.1 404 Not Found"


2026-08-31 11:56:26,614 [INFO] HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"


FAISS Index created with 1155 vectors saved to energy_knowledge_base.faiss


---
## Stage 5: Dynamic Query Construction & Forecast-Based Retrieval

In [10]:
def calculate_risk_level(gap_mu: float) -> str:
    if gap_mu < 3000.0:
        return "Low"
    elif 3000.0 <= gap_mu <= 4500.0:
        return "Moderate"
    else:
        return "High"


def build_forecast_query(forecast_result: Dict[str, Any]) -> str:
    month = forecast_result.get("month", "Target Month")
    demand = forecast_result.get("predicted_demand", 0.0)
    supply = forecast_result.get("predicted_supply", 0.0)
    gap = forecast_result.get("gap", demand - supply)
    risk = forecast_result.get("risk_level", calculate_risk_level(gap))
    
    if risk == "Low":
        query = (
            f"Tamil Nadu electricity demand supply planning for {month}. "
            f"Forecasted demand {demand:,.2f} MU, supply {supply:,.2f} MU, gap {gap:,.2f} MU (Low Risk < 3000 MU). "
            f"Measures for baseline grid stability, seasonal thermal maintenance scheduling, renewable energy integration, "
            f"resource adequacy planning, and demand-side management in Tamil Nadu."
        )
    elif risk == "Moderate":
        query = (
            f"Tamil Nadu electricity grid adequacy and power procurement for {month}. "
            f"Forecasted demand {demand:,.2f} MU, supply {supply:,.2f} MU, gap {gap:,.2f} MU (Moderate Risk 3000-4500 MU). "
            f"Strategies for short-term capacity expansion, peak load management, thermal hydro generation scheduling, "
            f"power purchase agreements (PPA), and demand-response mechanisms in Tamil Nadu."
        )
    else:
        query = (
            f"Tamil Nadu power deficit crisis mitigation and emergency energy planning for {month}. "
            f"Forecasted demand {demand:,.2f} MU, supply {supply:,.2f} MU, gap {gap:,.2f} MU (High Risk > 4500 MU). "
            f"Emergency load management protocols, inter-state grid power imports, fast-ramping generation deployment, "
            f"industrial demand control, and critical supply adequacy interventions in Tamil Nadu."
        )
    return query


def retrieve_relevant_chunks(forecast_result: Dict[str, Any], top_k: int = 5) -> Dict[str, Any]:
    query_str = build_forecast_query(forecast_result)
    
    query_vec = model.encode([query_str], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    faiss.normalize_L2(query_vec)
    
    scores, indices = index.search(query_vec, top_k)
    
    results = []
    for rank in range(top_k):
        idx = indices[0][rank]
        score = float(scores[0][rank])
        chunk = all_chunks[idx]
        results.append({
            "rank": rank + 1,
            "similarity_score": round(score, 4),
            "chunk_id": chunk["chunk_id"],
            "source": chunk["source"],
            "category": chunk["category"],
            "page": chunk["page"],
            "pages": chunk["pages"],
            "text": chunk["text"]
        })
        
    return {
        "generated_query": query_str,
        "retrieved_chunks": results
    }

---
## Stage 6: Gemini LLM Integration & Grounded Energy Recommendation

In Stage 6, we integrate the **Google Gemini LLM** to synthesize grounded energy planning recommendations based on the Top-K retrieved chunks from Stage 5.

### Strict Groundedness Rules:
1. **Zero Hallucination**: Every recommended measure MUST be anchored in retrieved Evidence Blocks.
2. **Exact Citations**: Inline citations format `[Document Name, Page N]`.
3. **Tamil Nadu Prioritization**: Tamil Nadu (`Tn`) evidence takes precedence for state operational decisions.
4. **Insufficient Evidence Clause**: Explicitly flags any policy area lacking retrieved context.

In [11]:
from generate_recommendation import generate_recommendation

# February 2026 Primary Test Case (Demand: 15,500.00 MU, Supply: 12,421.70 MU, Gap: 3,078.30 MU, Risk: Moderate)
feb_forecast = {
    "month": "February 2026",
    "predicted_demand": 15500.00,
    "predicted_supply": 12421.70,
    "gap": 3078.30,
    "risk_level": "Moderate"
}

retrieved_output = retrieve_relevant_chunks(feb_forecast, top_k=5)
rec_result = generate_recommendation(feb_forecast, retrieved_output["retrieved_chunks"])

print("=" * 90)
print("STAGE 6: GROUNDED ENERGY RECOMMENDATION RESULTS")
print("=" * 90)
print(f"Forecast Target     : {feb_forecast['month']}")
print(f"Predicted Gap       : {feb_forecast['gap']:,.2f} MU")
print(f"Risk Classification : {feb_forecast['risk_level']} Risk")
print(f"Execution Mode      : {rec_result['execution_mode']}")
print("=" * 90)
print("\n" + rec_result["recommendation"])
print("=" * 90)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-31 11:56:47,629 [INFO] AFC is enabled with max remote calls: 10.


2026-08-31 11:56:47,630 [WARNING] Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


2026-08-31 11:56:48,290 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.6-flash:generateContent "HTTP/1.1 429 Too Many Requests"


2026-08-31 11:56:48,294 [WARNING] Model gemini-3.6-flash attempt notice: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}}


...

2026-08-31 11:56:48,886 [INFO] AFC is enabled with max remote calls: 10.


2026-08-31 11:56:49,133 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent "HTTP/1.1 404 Not Found"


2026-08-31 11:56:49,136 [WARNING] Model gemini-1.5-flash attempt notice: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


2026-08-31 11:56:49,138 [WARNING] Online Gemini API call notice: All online model endpoints returned quota/availability notices.. Falling back to Grounded Offline Synthesis.


STAGE 6: GROUNDED ENERGY RECOMMENDATION RESULTS
Forecast Target     : February 2026
Predicted Gap       : 3,078.30 MU
Risk Classification : Moderate Risk
Execution Mode      : Grounded Deterministic Evidence Engine (Offline Mode)

GROUNDED ENERGY RECOMMENDATION REPORT
Notice: Online Gemini API Status: All online model endpoints returned quota/availability notices.
(Generated via Grounded Deterministic Evidence Synthesis Engine)

1. EXECUTIVE BRIEF & RISK DIAGNOSIS:
   - Forecast Period    : February 2026
   - Predicted Demand   : 15,500.00 MU
   - Predicted Supply   : 12,421.70 MU
   - Forecasted Deficit : 3,078.30 MU
   - Risk Classification: Moderate Risk (3,000 MU <= Gap <= 4,500 MU)

   The forecasted gap of 3,078.30 MU represents a MODERATE shortage risk requiring proactive
   grid balancing, peak market power purchases, and thermal outage control to maintain state supply adequacy.

2. PRIMARY STATE INTERVENTIONS (TAMIL NADU CONTEXT):
   - Power Market Optimization: Maximize gener